# Импорты

In [1]:
from collections.abc import Callable
from typing import List

import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Математическая модель

$
\bar{y} = \left( \begin{array}{c}
    r(t) \\
    v(t)
\end{array} \right)
$

Уравнение: <br>
$
\begin{cases}
    \displaystyle \frac{d\bar{y}}{dt} = \left( \begin{array}{c}
        v(t) \\
        (0.01 \pi)^2 r(t)
    \end{array} \right) \\
    \bar{y}(0) = \left( \begin{array}{c}
        0.04 \\
        0
    \end{array} \right)
\end{cases}
$

Решение: <br>
$
\bar{y} = \left( \begin{array}{c}
    0.02 (e^{0.01 \pi t} + e^{-0.01 \pi t}) \\
    0.0002\pi (e^{0.01 \pi t} - e^{-0.01 \pi t})
\end{array} \right)
$

In [2]:
omega = 0.01 * np.pi

y_0 = np.array([0.04, 0.0]) 

t_0 = 0.0
t_n = 60.0

def f(_t: float, y: List[float, float]) -> List[float, float]:
    r, v = y[0], y[1]
    dr_dt = v
    dv_dt = (omega ** 2) * r

    return np.array([dr_dt, dv_dt])

def y(t: float) -> List[float, float]:
    return np.array([
        0.02 * (np.exp(0.01 * np.pi * t) + np.exp(-0.01 * np.pi * t)),
        0.0002 * np.pi * (np.exp(0.01 * np.pi * t) - np.exp(-0.01 * np.pi * t))
    ])

# Решение

Решим уравнение методом Рунге-Кутта 4 порядка:<br>
$\displaystyle
\bar{y}_{i + 1} = \bar{y}_i + \frac{h}{6} (\bar{k}_1 + 2\bar{k}_2 + 2\bar{k}_3 + \bar{k}_4), \\
\bar{k}_1 = \bar{f}(t_i, \bar{y}_i), \\
\bar{k}_2 = \bar{f} \left( t_i + \frac{h}{2}, \bar{y}_i + \frac{h}{2} \bar{k}_1 \right), \\
\bar{k}_3 = \bar{f} \left( t_i + \frac{h}{2}, \bar{y}_i + \frac{h}{2} \bar{k}_2 \right), \\
\bar{k}_4 = \bar{f}(t_i + h, \bar{y}_i + h \bar{k}_3).
$

In [3]:
def runge_kutta_4(n: int, f: Callable[[List[float, float]], List[float, float]], t: List[float]) -> List[float, float]:
    h = (t_n - t_0) / n
    y_numeric = [y_0]

    for i in range(n):
        k_1 = f(t[i], y_numeric[i])
        k_2 = f(t[i] + h / 2, y_numeric[i] + h / 2 * k_1)
        k_3 = f(t[i] + h / 2, y_numeric[i] + h / 2 * k_2)
        k_4 = f(t[i] + h, y_numeric[i] + h * k_3)

        y_numeric.append(y_numeric[i] + h / 6 * (k_1 + 2 * k_2 + 2 * k_3 + k_4))

    return y_numeric

# Графики

In [11]:
def compare_solutions(n: int, f: Callable[[List[float, float]], List[float, float]], y: Callable[[float], List[float, float]]):
    t = np.linspace(t_0, t_n, n + 1)

    y_exact = np.array([y(t_i) for t_i in t])
    y_numeric = np.array(runge_kutta_4(n, f, t))

    r_exact = y_exact[:, 0]
    r_numeric = y_numeric[:, 0]
    r_error = np.abs(r_exact - r_numeric)

    figure = make_subplots(rows=1, cols=2)

    figure.add_trace(
        go.Scatter(x=t, y=r_exact, mode="lines", name="Точное", legend="legend"),
        row=1, col=1,
    )
    figure.add_trace(
        go.Scatter(x=t, y=r_numeric, mode="lines", name="Численное", legend="legend"),
        row=1, col=1,
    )

    figure.add_trace(
        go.Scatter(x=t, y=r_error, mode="lines", name="Погрешность", legend="legend2"),
        row=1, col=2,
    )

    figure.update_yaxes(exponentformat="power", row=1, col=1)
    figure.update_yaxes(exponentformat="power", row=1, col=2)
    figure.update_xaxes(title_text="t", row=1, col=1)
    figure.update_xaxes(title_text="t", row=1, col=2)
    figure.update_yaxes(title_text="r", row=1, col=1)
    figure.update_yaxes(title_text="error", row=1, col=2)

    figure.update_layout(
        height=600,
        legend=dict(
            orientation="v", 
            xanchor="left", 
            yanchor="top", 
            x=0,
        ),
        legend2=dict(
            orientation="v",
            xanchor="left",
            yanchor="top",
            x=0.55
        )
    )
    figure.show()

ns = [10, 100, 1000]
for n in ns:
    compare_solutions(n, f, y)